<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/main/colabs/cross_app_authz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Okta Cross-App Access Demo

This notebook demonstrates the complete **Identity Assertion Authorization Grant (ID-JAG)** flow for secure cross-application access using the Okta AI SDK.


## The 5-Step Cross App Access Flow

1. **Get Access Token** - Authenticate as a user to obtain an access token
1. **Exchange Access Token for ID-JAG Token** - Convert user's Access token to a JWT Assertion Grant Token
1. **Verify ID-JAG Token** - Validate the token's authenticity and claims
1. **Exchange ID-JAG for new Access Token** - Get an access token for the target resource
1. **Verify Access Token** - Validate final access token before use

---

### Prerequisites

Before running this notebook, you need:

1. **Okta Organization** with:
   - Custom authorization server configured
   - OAuth 2.0 application with Token Exchange enabled
   - ID-JAG support enabled

2. **Agent/Workload Principal** with:
   - Principal ID (agent identifier)
   - Private JWK (RSA key pair for JWT bearer assertion)

3. **User**:
   - Valid Okta User to authenticate and obtain an access token.

## Setup and Installation


### 1. Install the Okta Python SDK

In [ ]:
# Install the Okta AI SDK from PyPI

%pip install --upgrade okta-client-python

### 2. Configuration Variables

Set _all_ your configuration variables and _run the cell._

In [ ]:
# @title { display-mode: "form", vertical-output: true }
# @markdown _Enter your configuration variables here and click the 'run' to validate._
# @markdown <br><br> These will be used throughout the notebook.

OKTA_DOMAIN = 'https://atko-ai.oktapreview.com' # @param {"type":"string","placeholder":"Enter your entire Okta domain (including https)"}
# @markdown <br>

# @markdown ---
# @markdown ##### Client (App) configuration
# @markdown ---
# CLIENT_ID = '' # @param {"type":"string","placeholder":"Enter your client Id"}
CLIENT_ID = '0oa12sbtk7dJkniEU1d8' # @param {"type":"string","placeholder":"Enter your client Id"}
REDIRECT_URI = 'http://localhost:8080/authorization-code/callback' # @param {"type":"string","placeholder":"Enter application redirect URI"}
# CLIENT_AUTHZ_SERVER_ID = '' # @param {"type":"string","placeholder":"Enter your authorization server Id"}
CLIENT_AUTHZ_SERVER_ID = 'aus12sbpbhfkq37jB1d8' # @param {"type":"string","placeholder":"Enter your authorization server Id"}
CLIENT_SCOPES = ['openid', 'profile', 'email', 'mcp:read'] # @param {"type":"raw","placeholder": "Enter the scopes your client should request."}

# @markdown <em>If you are using client secret authentication, enter your client secret below.</em>
# CLIENT_SECRET = '' # @param {"type":"string","placeholder":"Enter your client secret"}
CLIENT_SECRET = 'zdzkaJt7xTvBnzTbvFF0Ru3xRTFUuGYFcDZSrRCvkbaxOnfD7vetlQFOGNowGsVb' # @param {"type":"string","placeholder":"Enter your client secret"}
# @markdown <br><em>If you are using private key authentication, enter your application's private JWK below.</em>
CLIENT_PRIVATE_JWK = {} # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Principal/Agent Configuration
# @markdown ---
# PRINCIPAL_ID = '' # @param {"type":"string","placeholder":"Enter your agent identifier"}
PRINCIPAL_ID = 'wlp12sbrnlc316g7W1d8' # @param {"type":"string","placeholder":"Enter your agent identifier"}
RESOURCE_URI = 'https://api' # @param {"type":"string","placeholder":"Enter the audience/resource URL configured on the agent."}
PRINCIPAL_SCOPES = ['mcp:read'] # @param {"type":"raw","placeholder": "Enter the scopes your agent should request."}

# @markdown <em>If you are using client secret authentication, enter your agent secret below.</em>
PRINCIPAL_SECRET = '' # @param {"type":"string","placeholder":"Enter your agent's client secret"}
# @markdown <br><em>If you are using private key authentication, enter your agent's private JWK below.</em>
PRINCIPAL_PRIVATE_JWK = {} # @param {"type":"raw","placeholder": "{}"}

# PRINCIPAL_PRIVATE_JWK = {
#   "alg": "RS256",
#   "d": "Q2tnDZ5cmqdHuO1pIRO4WPJmpQwkZA2g3R21tPHCbvryIdf3xbPEkwmxcogo5EbOIiBmsHay8eNYL1mTRrOtgfr1UpzezcvI9nGoKgC0VCI5h5kC5t6yXTENZYqGMndgejiJ7fN89FwVixbFK8RY2SaXpq0wtUMG7uIBFCiHv4sn-ePcUpr09mlEH-G87NC_ENuBDZZ2e1dft8QLuO25b0VsLj99PZEueqJet7P0P9FzGXEk1oaE7EoDpDPy5brnt0gMjOnYI6zMW2eFbyuTopky4RzBtAmjyIO8SYAp-K0eyZV59OmzwBeJFRY9YvJ0-6LQSzRi_r0qxJDc0x7vpw",
#   "dp": "UoBSQJC6lpxGD6eWSR3mKhSjY8lhb7eR9wijwvL5eVIcjXC0RjnPH6G4ie6w1TJrvrJG56hdR7YaDOf8BgEIySBrL46u5edbP2zLN6WMg03K73c6KnO2QaWwhjEDh-395EreTLF2Zz_Jnw7pIkFYQtQSmn1x-SAEx4lkIGQ8L78",
#   "dq": "qg0e_Wx0_ICHoL1bXNBiBV8bXabVS3bDIP41_z4la75L6j6kZOe0K624wAHJfKDzL1XcP3ir9Wr_DEpB5K17C-W6XkfnK5ic5EBkcgqqEcfNOPSA-i4rqnthVPiSqwyPdna7QJFkBpzrIl_dJNrlTYYWEi2xfzLHJtwJpAXBq4M",
#   "e": "AQAB",
#   "kty": "RSA",
#   "n": "lKCn5N6V_bv3M3mYijNuHRqZ2kxMsZCAKMVKilpEO8HACKEdDBljbaUuYL6Cuzts8eC46JXDFJIMNUjRZNeB4F_1O3MUig-VlX4as1qzdRZ8RIQREQZ7Y83IvyKM6idjL3YMf8lB40XtTD4W5ugscnBbq1L1vlqin3rnaXRZraoJtuAVTbv38hPLIT_4pefZd7031R_eq_4SivDAl-a0rpVetp1ZtwdwvGWX4I5sYemwISsrGIqf1mJVIKkHBa7f1bPxhGcBBAQ-S05kHAcOUe4IIjNpd0yq5COW9w-dVRS3J_Zy97Wy6VENP6TEL0FvwG7-LknN82ATU734NDe55Q",
#   "p": "w-_m7FBeUqXcd9_rX9fEoHHTDCHan6rQs7Oxxaw7czUvIetT_mL2IiD44ZxMJRnrZZMVdQqrA4KguY9K5LndFSwPQFEHqLSUPGAY-la96LLrVUkNpL_3lsw-tjJj7ClH-7PXScvyiU1QTPef5pEZBZQfC9Qkds0-kcyx4EHfU3s",
#   "q": "wjAnFiK5Ls_pGIrVRbtdEefRWF7UCrStDbIWDQjbeK5bBqmGTZawd_tGDZANxHGRGeF6f9GpEleHT11pLm2K_GmjSAkyA8piqgAVWbsDLGn2V3Z6ffAHvx7PCYgtTOuM4C7l1dUC12h0FIzUL7qaK49OCja7AKObyS8liVXseh8",
#   "qi": "U9eJa5hrBZbbKovy54-DGEMl_0TsPo1ZTtEUkmg2lu7TYBC9AM65RSupqzxkLHTI_rQfpy_zgy8hpKaq4aRsgHzpdthqE-h4vtw7jKFYeJ03w-ahCs3clAYUZYoya5QBc5jC-MV5VUSI9UDczux3Jovczb__z01QGHnBeprqtsE",
#   "kid": "8fdcbc8c32acefacee427a949824bc9d",
#   "use": "sig"
#   } # @param {"type":"raw","placeholder": "{}"}
# @markdown <br>

# @markdown ---
# @markdown ##### Resource Server Configuration
# @markdown ---

RESOURCE_AUTHZ_SERVER_ID = 'ausubr9dq7o80RBml1d7' # @param {"type":"string","placeholder":"Enter your authorization server Id"}
RESOURCE_SERVER_AUDIENCE = 'https://benefits.streamward.com' # @param {"type":"string","placeholder":"Enter your resource server audience"}

# @markdown <br>
SDK_DEBUG_ENABLED = True # @param {"type":"boolean"}

# Import utility functions for validation and other operations
# import requests
from enum import Enum
from typing import Callable

# ===== Only for local dev =====
from pathlib import Path
import sys

cwd = Path.cwd()
repo_root = cwd if (cwd / "utils.py").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import utils
validate_config = utils.validate_config
# ==============

# url = "https://raw.githubusercontent.com/udplabs/okta-ai-poc/refs/heads/main/utils.py"

# response = requests.get(url)
# if response.status_code == 200:
#     # This executes the code inside utils.py
#     exec(response.text)
#     print("Successfully loaded utils.py from GitHub!")
# else:
#     raise Exception(f"Failed to load utils.py. Please contact a code owner -- this is not good!")

# validate_config: Callable[[dict, str], None] | None = None

# Perform validation of the configuration variables
# Function will always exist in utils.py, but we check for its existence to avoid errors if the file fails to load.
if validate_config:
    try:
        validate_config(locals(), "authz")
    except ValueError as e:
        print(e)
        raise SystemExit("❌ Configuration validation failed. Please fix the above errors and re-run the cell.")

print("✅ All configuration variables validated successfully!")
print(f"    Okta Domain: {OKTA_DOMAIN}")
print(f"    Client ID: {CLIENT_ID}")
print(f"    Client Authorization Server: {CLIENT_AUTHZ_SERVER_ID}")

print(f"    Principal ID: {PRINCIPAL_ID}")
print(f"    Resource URI: {RESOURCE_URI}")
print(f"    Resource Authorization Server: {RESOURCE_AUTHZ_SERVER_ID}")
print(f"    Resource Server Audience: {RESOURCE_SERVER_AUDIENCE}")

global RESOURCE_ISSUER
global CLIENT_ISSUER
RESOURCE_ISSUER = f"{OKTA_DOMAIN}/oauth2/{RESOURCE_AUTHZ_SERVER_ID}"
CLIENT_ISSUER = f"{OKTA_DOMAIN}/oauth2/{CLIENT_AUTHZ_SERVER_ID}"


✅ All configuration variables validated successfully!
    Okta Domain: https://atko-ai.oktapreview.com
    Client ID: 0oa12sbtk7dJkniEU1d8
    Client Authorization Server: aus12sbpbhfkq37jB1d8
    Principal ID: wlp12sbrnlc316g7W1d8
    KID: 8fdcbc8c32acefacee427a949824bc9d
    Resource URI: https://api
    Resource Authorization Server: ausubr9dq7o80RBml1d7
    Resource Server Audience: https://benefits.streamward.com


### 3. Initialize the SDK for _user authentication_

In [ ]:
# Initialize Okta SDK
from okta_client.authfoundation import OAuth2Client, OAuth2ClientConfiguration, ClientSecretAuthorization, LocalKeyProvider
from okta_client.authfoundation.oauth2.jwt_bearer_claims import JWTBearerClaims
from okta_client.authfoundation.oauth2.client_authorization import ClientAssertionAuthorization
from okta_client.oauth2auth import AuthorizationCodeContext, AuthorizationCodeFlow, CrossAppAccessFlow, CrossAppAccessTarget, Prompt

print("✅ Imports successful!")

client_authz = None;

# Determine the client authentication method based on the provided configuration and set up the appropriate authorization flow.
if CLIENT_PRIVATE_JWK and isinstance(CLIENT_PRIVATE_JWK, dict) and CLIENT_PRIVATE_JWK.get("kid"):
    print("\n✅ Client private JWK is provided. Initializing Client Assertion Authorization for the client application...")
    client_authz = ClientAssertionAuthorization(
        assertion_claims=JWTBearerClaims(
            issuer=CLIENT_ID,
            subject=CLIENT_ID,
            audience=CLIENT_ISSUER,
            expires_in=300  # Token expiration time in seconds
        ),
        key_provider=LocalKeyProvider(
            key=CLIENT_PRIVATE_JWK,
            algorithm=CLIENT_PRIVATE_JWK.get("alg", "RS256"),
            key_id=CLIENT_PRIVATE_JWK.get("kid")
        )
    )
else:
    client_authz = ClientSecretAuthorization(
        id=CLIENT_ID,
        secret=CLIENT_SECRET
    )

# Initialize the user SDK with the appropriate configuration, including the issuer for the custom authorization server.
user_sdk_config = OAuth2ClientConfiguration(
    issuer=CLIENT_ISSUER,
    scope=CLIENT_SCOPES,
    redirect_uri=REDIRECT_URI,
    client_authorization=client_authz
)

global user_sdk
user_sdk = OAuth2Client(configuration=user_sdk_config)

print("✅ User SDK initialized!")

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("[WARN] Debugger not loaded; continuing without SDK listener.")
    else:
        user_sdk.listeners.add(debugger_cls()) # type: ignore


✅ Imports successful!
✅ User SDK initialized!
[WARN] Debugger not loaded; continuing without SDK listener.


---

## Step 1: Obtain User Tokens

If you don't have user tokens yet, use this section to obtain them via OIDC.

### Important:
This step uses a **Custom Authorization Server**. 

This is the correct approach for obtaining the initial user token that will be used in the ID-JAG flow.

### The Flow:

1. **Build authorization URL** - Users authenticate using OIDC via the browser.
2. **Copy redirect URL from browser** - Copy the **entire** redirect URL after authentication.
3. **Exchange code for tokens** - Get ID token with issuer = Okta domain

### 1.1 Build and Open Authorization URL

Run the following cell and then click the generated button to open the authorization URL in a new browser tab.

In [ ]:
from IPython.display import HTML, display # Ensure display is imported for the HTML button

async def authorize():

  try:
    # Step 1: Build the authorization URL
    authorization_context = AuthorizationCodeContext(
        prompt=Prompt.LOGIN,
        # The `resource` parameter is used (per RFC8693) to specify the audience/resource for which the access token is requested. In this case, it is set to the RESOURCE_URI defined earlier.
        resource=RESOURCE_URI
    )

    global auth_flow
    auth_flow = AuthorizationCodeFlow(client=user_sdk)

    authorization_url = await auth_flow.start(context=authorization_context)

    print("✅ Authorization URL generated!")
    print(f"\n{authorization_url}")
    print("\n" + "="*80)

    # Display clickable button
    html_button = f"""
    <div style="margin: 20px 0;">
        <a href="{authorization_url}" target="_blank" style="
            display: inline-block;
            padding: 15px 30px;
            background-color: #007bff;
            color: white;
            text-decoration: none;
            border-radius: 5px;
            font-weight: bold;
            font-size: 16px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.2);
        ">Click Here to Authenticate with Okta</a>
    </div>
    <div style="margin: 20px 0; padding: 15px; background-color: #fff3cd; border-left: 4px solid #ffc107; border-radius: 4px;">
        <strong>Instructions:</strong>
        <ol style="margin: 10px 0 0 0;">
            <li>Click the button above to open the authorization URL in a new tab</li>
            <li>Sign in with your Okta credentials</li>
            <li>After authentication, you'll be redirected to: <code>{REDIRECT_URI}</code></li>
            <li>Copy the <strong>code</strong> parameter from the URL (it will look like: <code>?code=ABC123...</code>)</li>
            <li>Paste the code in the next cell to exchange it for tokens</li>
        </ol>
    </div>
    """

    display(HTML(html_button))

    print("\nWhat to do next:")
    print("   1. Click the button above")
    print("   2. Sign in to Okta")
    print("   3. Copy the entire URL")
    print("   4. Paste it in the next cell")
    print("\nNote: The tokens will be issued by the Client Authorization Server")
    print(f"      Issuer: {CLIENT_ISSUER}")

  except Exception as e:

    print(f"❌ [ERROR]: {e}")
    raise

await authorize()

✅ Authorization URL generated!

https://atko-ai.oktapreview.com/oauth2/aus12sbpbhfkq37jB1d8/v1/authorize?client_id=0oa12sbtk7dJkniEU1d8&request_uri=urn%3Aokta%3AOEFUZFpHR04teU9laGNkT2lWajhvclZyeFY2Vi1OZERfUFhISDN1RzBIazo




What to do next:
   1. Click the button above
   2. Sign in to Okta
   3. Copy the entire URL
   4. Paste it in the next cell

Note: The tokens will be issued by the Client Authorization Server
      Issuer: https://atko-ai.oktapreview.com/oauth2/aus12sbpbhfkq37jB1d8


### 1.2 Exchange Authorization Code for Tokens

After authenticating...
1. copy the entire URL
1. paste the URL below
1. and run this cell to obtain tokens

In [ ]:
# @title { display-mode: "form" }

# REDIRECT_URL = '' # @param { type: "string", placeholder: "Insert entire URL string here."}
REDIRECT_URL = 'http://localhost:8080/authorization-code/callback?code=45gOs-C5HVql2331CzwHYesV9rgdFMflmZ9fBSvgeG4&state=7cbd4749-62e2-4003-96a4-9ac62e7681ba' # @param { type: "string", placeholder: "Insert entire URL string here."}

async def exchange_code_for_tokens():
  if not REDIRECT_URL:
    print("\n❌ No url provided! Please paste the entire URL containing the `code` and run this cell again.")
  else:
    print("\n" + "=" * 80)
    print("Exchanging Authorization Code for User Tokens...")
    print("=" * 80)

    try:

      token = await auth_flow.resume(REDIRECT_URL)

      print("Token exchange successful!\n")
      print("Token Response:")
      print(f"   Token Type: {token.token_type}")
      print(f"   Expires In: {token.expires_in} seconds")
      print(f"   Scope: {token.scope}")

      # Extract tokens
      global ID_TOKEN, ACCESS_TOKEN, REFRESH_TOKEN
      ID_TOKEN = token.id_token.raw
      ACCESS_TOKEN = token.access_token
      REFRESH_TOKEN = token.refresh_token

      print(f"\nTokens Obtained:")
      if ID_TOKEN:
          print(f" ✅ ID Token: https://jwt.io#token={ID_TOKEN}")
      if ACCESS_TOKEN:
          print(f" ✅ Access Token: https://jwt.io#token={ACCESS_TOKEN}")
      if REFRESH_TOKEN:
          print(f" ✅ Refresh Token: {REFRESH_TOKEN[:50]}...")

      # Decode and display access token claims (optional)
      if ACCESS_TOKEN:
          import jwt
          decoded = jwt.decode(ACCESS_TOKEN, options={"verify_signature": False})
          print(f"\nAccess Token Claims:")
          print(f"   Subject: {decoded.get('sub')}")
          print(f"   Email: {decoded.get('email', 'N/A')}")
          print(f"   Name: {decoded.get('name', 'N/A')}")
          print(f"   Issuer: {decoded.get('iss')}")
          print(f"   Audience: {decoded.get('aud')}")

          # Verify issuer is the Okta domain.
          if decoded.get('iss') == CLIENT_ISSUER:
              print(f"\n ✅ Token issued by Custom Authorization Server: {CLIENT_ISSUER}")
          else:
              print(f"\n ⚠️ Unexpected issuer: {decoded.get('iss')}")
              print(f"   Expected: {CLIENT_ISSUER}")

      print("\n" + "="*80)
      print("You can now use the ACCESS_TOKEN variable in the cross-app access flow!")
      print("="*80)

      print("Now verifying configuration...")

      # Verify configuration is set
      print("Cross-App Access Configuration")
      print("=" * 60)

      print(f"\nConfiguration Summary:")
      print(f"   Okta Domain: {OKTA_DOMAIN}")

      print(f"   Client ID: {CLIENT_ID}")
      print(f"   Client Auth Server: {CLIENT_AUTHZ_SERVER_ID}")

      print(f"\n   Principal ID: {PRINCIPAL_ID}")
      print(f"    Principal KID: {PRINCIPAL_PRIVATE_JWK.get('kid')}")
      print(f"    Resource URI: {RESOURCE_URI}")

      print(f"\n   Resource Auth Server: {RESOURCE_AUTHZ_SERVER_ID}")
      print(f"   Resource Server Audience: {RESOURCE_SERVER_AUDIENCE}")
      print("=" * 60)

    except Exception as e:
        print(f" ❌ Error during token exchange: {e}")
        print("\nTroubleshooting:")
        print("   • Make sure you copied the entire authorization code")
        print("   • Verify your redirect_uri matches what's registered in Okta")
        print("   • Check that the authorization code hasn't expired (valid for ~60 seconds)")
        print("   • Ensure your client_id and client_secret are correct")

await exchange_code_for_tokens()

---

---

## Step 2: Exchange Access Token for ID-JAG Token

The first step is to exchange a user's Okta access token for an ID-JAG token. This token represents the user's identity and can be used for cross-application access.

### What happens:
The SDK...
1. generates a JWT bearer assertion using your private key;
1. calls the resources **custom authorization server's** token endpoint;
1. exchanges access token for ID-JAG token with specified audience;
1. exchanges the ID-JAG for an access token;
1. validates the returned access token;
1. returns access token.

### 2.1 Initialize the Okta _agent_ SDK

First, let's initialize a new instance of the SDK with cross-app access configuration for our *agent* specifically.

In [ ]:

client_authz = None;

if PRINCIPAL_PRIVATE_JWK and isinstance(PRINCIPAL_PRIVATE_JWK, dict) and PRINCIPAL_PRIVATE_JWK.get("kid"):
    print("\n✅ Principal private JWK is provided. Initializing Client Assertion Authorization for the agent...")
    client_authz = ClientAssertionAuthorization(
        assertion_claims=JWTBearerClaims(
            issuer=PRINCIPAL_ID,
            subject=PRINCIPAL_ID,
            audience=f"{OKTA_DOMAIN}/oauth2/v1/token",
            expires_in=300  # Token expiration time in seconds
        ),
        key_provider=LocalKeyProvider(
            key=PRINCIPAL_PRIVATE_JWK,
            algorithm=PRINCIPAL_PRIVATE_JWK.get("alg", "RS256"),
            key_id=PRINCIPAL_PRIVATE_JWK.get("kid")
        )
    )
else:
    client_authz = ClientSecretAuthorization(
        id=PRINCIPAL_ID,
        secret=PRINCIPAL_SECRET
    )

# The SDK will use the key and claims to generate a signed JWT for client assertion (or use the client_id and client_secret if configured) when requesting an access token from the Okta authorization server.
# This is how the AGENT authenticates with Okta in order to perform the token exchange and obtain an access token for the resource server.
agent_sdk_config = OAuth2ClientConfiguration(
    issuer=OKTA_DOMAIN,
    client_authorization=client_authz,
    scope=PRINCIPAL_SCOPES
)

print("✅ OAuth2 client configuration created")

# Create OAuth2 client
agent_sdk = OAuth2Client(configuration=agent_sdk_config)

if SDK_DEBUG_ENABLED:
    debugger_cls = globals().get("Debugger")
    if debugger_cls is None:
        print("⚠️ WARN: Debugger not loaded; continuing without SDK listener.")
    else:
        agent_sdk.listeners.add(debugger_cls()) # type: ignore

print("✅ OAuth2 client created")

print("✅ Agent SDK initialized successfully!")

✅ Key provider created
✅ JWT bearer claims created
✅ OAuth2 client configuration created
[WARN] Debugger not loaded; continuing without SDK listener.
✅ OAuth2 client created
✅ Agent SDK initialized successfully!


### 2.2 Exchange Token
Great! Now let's start the token exchange.

In [ ]:
print("\n" + "=" * 80)
print("STEP 2.1: Exchange user's access token for an ID-JAG")
print("=" * 80)

# Create target
global agent_sdk_target
agent_sdk_target = CrossAppAccessTarget(
    issuer=RESOURCE_ISSUER
)

print(f"✅ Target created: {agent_sdk_target.issuer}")

# Create cross-app flow
global agent_sdk_flow
agent_sdk_flow = CrossAppAccessFlow(
    client=agent_sdk,
    target=agent_sdk_target
)

print("✅ Cross-app access flow created")

async def exchange_id_token():
  try:

    id_jag_result = await agent_sdk_flow.start(
       token=ACCESS_TOKEN,
       token_type="access_token",
       audience=RESOURCE_SERVER_AUDIENCE,
       scope=PRINCIPAL_SCOPES
    )

    # result.resume_assertion_claims is None \u2192 fully automatic
    assert id_jag_result.resume_assertion_claims is None

    # Store for next step
    global id_jag_token
    id_jag_token = agent_sdk_flow.context.id_jag_token

    print("\n✅ ID-JAG token obtained!")
    print(f"\nToken Details:")
    print(f"   Token Type: {id_jag_token.issued_token_type}")
    print(f"   Expires In: {id_jag_token.expires_in} seconds")
    print(f"   Scope: {id_jag_token.scope or 'N/A'}")
    print(f"   Decoded Token: https://jwt.io#token={id_jag_token.access_token}")

  except Exception as e:
      print(f"❌ [ERROR]: {e}")
      raise

await exchange_id_token()

---

## Step 3: Verify ID-JAG Token

Before using the ID-JAG token, we should verify its authenticity. This step:
- Validates the token signature using Okta's public keys (JWKS)
- Checks the audience, issuer, and expiration claims
- Extracts user information from the token

***SECURITY NOTE:*** *Always verify tokens before trusting their contents.* 

This prevents:
- Tampered tokens
- Expired tokens
- Tokens meant for different audiences

In [ ]:
import time
from datetime import datetime
print("\n" + "=" * 80)
print(f"Step 3: Verify ID-JAG token")
print("=" * 80)

print(f"   Expected Audience: {agent_sdk_target.issuer}\n")

def decode_token(token: str) -> dict:
    """Decode JWT without verification (for display purposes)"""
    import jwt
    return jwt.decode(token, options={"verify_signature": False})

decoded_jag = decode_token(id_jag_token.access_token)

# Verify audience
token_aud = decoded_jag.get('aud')
if isinstance(token_aud, list):
    aud_match = agent_sdk_target.issuer in token_aud
else:
    aud_match = token_aud == agent_sdk_target.issuer

if aud_match:
    print(f"✅ Audience matches: {agent_sdk_target.issuer}")
else:
    print(f"⚠️  Audience mismatch! Expected: {agent_sdk_target.issuer}, Got: {token_aud}")

# Check expiration
exp = decoded_jag.get('exp')
if exp and exp > time.time():
    exp_time = datetime.fromtimestamp(exp)
    print(f"✅ Token valid until: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⚠️  Token expired or no expiration!")

---
## Step 4: Exchange ID-JAG Token for Authorization Server Token

Now we exchange the verified ID-JAG token for an access token from a custom authorization server. This token can be used to access protected resources.

### What happens:
1. SDK generates a JWT bearer assertion for the authorization server (*or uses client_id/client_secret if configured*)
2. Uses the ID-JAG token as the assertion
3. Calls the custom authorization server's token endpoint
4. Returns an access token with appropriate scopes

### Why this matters:
This is useful when you need to access resources protected by a custom authorization server with specific scopes and policies.

In [ ]:
async def exchange_id_jag_token():
  try:

    print("\n" + "=" * 80)
    print(f"Step 4: Exchange ID-JAG for resource access token")
    print("=" * 80)

    global auth_server_token
    auth_server_result = await agent_sdk_flow.resume()
    print(auth_server_result)

    print("✅ Authorization server token obtained!")
    print(f"Token Details:")
    print(f"   Token Type: {auth_server_result.token_type}")
    print(f"   Expires In: {auth_server_result.expires_in} seconds")
    print(f"   Scope: {auth_server_result.scope or 'N/A'}")
    print(f"   Decoded Token: https://jwt.io#token={auth_server_result.access_token}")

    if auth_server_result.refresh_token:
        print(f"   Refresh Token: Available")

    # Store for next step
    global auth_server_token
    auth_server_token = auth_server_result.access_token


  except Exception as e:
      print(f"[ERROR] Error: {e}")
      raise

await exchange_id_jag_token()

---
## Step 5: Verify Access Token

The final step is to verify the authorization server token before using it to access resources.

### What happens:
1. Validates the token signature using the authorization server's public keys
2. Checks audience, issuer, and expiration
3. Extracts scope and user information

### Important:
The resource server (*your API*) should perform this verification on every request to ensure:
- Token is valid and not tampered with
- Token is not expired
- Token is intended for this resource server (*audience check*)
- User has required permissions (*scope check*)

In [ ]:
print("\n" + "=" * 80)
print("STEP 5: Verify Access Token")
print("=" * 80)

decoded_access = decode_token(auth_server_token)

# Verify issuer
expected_issuer = RESOURCE_ISSUER
if decoded_access.get('iss') == expected_issuer:
    print(f"✅ Issuer matches: {expected_issuer}")
else:
    print(f"⚠️  Issuer mismatch! Expected: {expected_issuer}, Got: {decoded_access.get('iss')}")

# Verify audience
token_aud = decoded_access.get('aud')
if isinstance(token_aud, list):
    aud_match = RESOURCE_SERVER_AUDIENCE in token_aud
else:
    aud_match = token_aud == RESOURCE_SERVER_AUDIENCE

if aud_match:
    print(f"✅ Audience matches: {RESOURCE_SERVER_AUDIENCE}")
else:
    print(f"⚠️  Audience mismatch! Expected: {RESOURCE_SERVER_AUDIENCE}, Got: {token_aud}")

# Check expiration
exp = decoded_access.get('exp')
if exp and exp > time.time():
    exp_time = datetime.fromtimestamp(exp)
    print(f"✅ Token valid until: {exp_time.strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⚠️  Token expired or no expiration!")

# Check scopes
scope_claim = decoded_access.get('scp', decoded_access.get('scope', ''))
if isinstance(scope_claim, list):
    scopes = scope_claim
elif isinstance(scope_claim, str):
    scopes = scope_claim.split()
else:
    scopes = []

if scopes:
    print(f"\n✅ Granted Scopes: {', '.join(scopes)}")
    if 'mcp:read' in scopes:
        print("✅ Has 'mcp:read' permission")

print(decoded_access.get('cid'))
print(decoded_access.get('sub'))

**Note** 
The ID JAG is a short lived one time use token. Access token contains the sub ("human in the loop") and cid ("AI agent workload principal").  Review syslog for audit log.  

##### Congratulations on successful completion of the cross app access flow and securing your AI Agents with Okta!

---

## Resources

- [Okta AI SDK Documentation](https://github.com/okta/okta-client-python/tree/main)
- [ID-JAG - Identity Assertion Authorization Grant](https://datatracker.ietf.org/doc/draft-ietf-oauth-identity-assertion-authz-grant/)

---
